# Notebook to clean traffic stop data

In [260]:
# install packages if necessary
%pip install -r "../requirements.txt"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [261]:
import pandas as pd
import geopandas as gpd
import zipfile
import numpy as np

# Function to load data
- Loads in police stop data from the [Stanford Opening Policing Project](https://openpolicing.stanford.edu/data/)

- Currently data is available for several states throughout the U.S.
- For our objectives, we downloaded data for San Francisco. 
- This data has observations collected between December of 2006 to June of 2016.


In [262]:
def load_data(path, lat = None, long = None, geospatial = False):
    # path is the string of the path to datafile
    # lat/long are strings for the name of the latitude and longitude columns in the datafile
    # set geospatial to true if trying to extract xy coordinates from a csv file
    
    # unzip data if name ends with .zip
    if(path[-3:]=="zip"):
        with zipfile.ZipFile(path) as z:
            csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
            df = pd.read_csv(z.open(csv_file))
        print("Unzipped")
    # read in regularly for non zipped files
    else:
        # split into last file name after /
        if("geojson" in path.rsplit('/', 1)[-1]):
            df = gpd.read_file(path)
        else:
            df = pd.read_csv(path)

    # convert to a geospatial dataframe to extract x,y coordinates
    if(geospatial):
        df = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[long], df[lat]),
            # crs set for lat/long data
            crs="EPSG:4326" 
        )

        # Print coordinate reference system to verify correct locations
        print("Data CRS:", df.crs)
    
    return df

# Filter by location
- Since we are only interested in stops occuring in San Francisco county, we filter out any data from outside the county boundaries

In [263]:
def filter_data_loc(df, county_df, counties_of_interest):
    # df is data to filter
    # county_df is a geospatial dataframe containing polygons that define the county of interest
    # counties of interest is a list of strings with each string corresponding to a county in the county_df that is necessary to keep

    # filter points to within bounaries of county polygon
    boundaries = county_df[county_df['county'].isin(counties_of_interest)]
    filtered_df = gpd.sjoin(df, boundaries, predicate="within")
     
    return filtered_df

If necessary, change the local path names `stops_path` and `county_path`, counties of interest (`counties_of_interest`), and column names for data (`lat` and `long` arguments in the `load_data` function)

Additionally, the dates to filter between can be changed.

# Filter by time
- Since we are only interested in stops within a few years of the Vision Zero implementation, we filter data to keep observations between the period of January of 2010 to June of 2016

In [264]:
def filter_data_time(df, dt_col, start, stop):
    # check values are in datetime format
    df[dt_col] = pd.to_datetime(df[dt_col])

    start = pd.to_datetime(start)
    stop = pd.to_datetime(stop)
    # filter
    df = df[df[dt_col].between(start, stop)]

    return df

In [265]:
# change path names if necessary
stop_path = r"../data/sf_police_stops_raw.csv.zip"
county_path = r"../data/Bay_Area_County_Polygons.geojson"

# change to different counties of interest if necessary
counties_of_interest = ['San Francisco']

# load crash, county data
# change latitude and longitude column names if necessary
stops_df = load_data(stop_path, lat = "lat", long = "lng", geospatial = True)
county_df = load_data(county_path)

# filter to coordinates within relevant counties
stops_df = filter_data_loc(stops_df, county_df, counties_of_interest)

# filter to relevant dates
stops_df = filter_data_time(stops_df, 'date', '1-01-2010', '06-01-2016')


/var/folders/6l/_5f2v6_x5hj1cd_lrwrm8k9r0000gn/T/ipykernel_9982/2371665872.py:10: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(csv_file))


Unzipped
Data CRS: EPSG:4326


Since these operations are necessary for processing other datasets, these functions have also been exported to the `data_functions.py` file for use in other notebooks.

# Features and feature engineering
- For analyzing stop data, we are interested in certain features
- Here we restrict data to our relevant predictor and outcome variables

In [266]:
var = ['date', 'time',
 'lat', 'lng',
 'subject_age', 'subject_race', 'subject_sex',
 'arrest_made', 'citation_issued', 'warning_issued',
 'search_conducted', 'search_vehicle', 
 'reason_for_stop']

stops_df = stops_df[var]

# get number of rows and columns for observations
stops_df.shape

(568996, 13)

In [267]:
# check distinct values for columns 
col_values = ['subject_race', 'type', 'reason_for_stop']
for column_name in stops_df.loc[:, stops_df.columns.isin(col_values)]:
    print(f"Unique Values for {column_name}")
    print(pd.unique(stops_df[column_name].values.ravel()))
    print()

Unique Values for subject_race
['asian/pacific islander' 'black' 'hispanic' 'white' 'other']

Unique Values for reason_for_stop
['Mechanical or Non-Moving Violation (V.C.)' 'Moving Violation'
 'MPC Violation' 'DUI Check' nan 'Traffic Collision'
 'Assistance to Motorist' 'BOLO/APB/Warrant'
 'Moving Violation|Mechanical or Non-Moving Violation (V.C.)'
 'DUI Check|MPC Violation' 'Moving Violation|NA'
 'Mechanical or Non-Moving Violation (V.C.)|DUI Check'
 'Moving Violation|Traffic Collision' 'Moving Violation|MPC Violation'
 'Moving Violation|DUI Check'
 'Mechanical or Non-Moving Violation (V.C.)|BOLO/APB/Warrant'
 'Moving Violation|Mechanical or Non-Moving Violation (V.C.)|Mechanical or Non-Moving Violation (V.C.)|Mechanical or Non-Moving Violation (V.C.)|Mechanical or Non-Moving Violation (V.C.)'
 'Moving Violation|Assistance to Motorist'
 'MPC Violation|Moving Violation'
 'Mechanical or Non-Moving Violation (V.C.)|Moving Violation'
 'NA|Moving Violation']



Based on the values in the column indicating the reason for a stop, we want to simplify these outcomes into binary flags indicating whether or not a reason for a traffic stop was selected. 
This simplifies the number of outcomes from 20 to 7:
These include Moving Violations, Mechanical or Non-Moving Violations, DUI Checks, MPC Violations, Traffic Collisions, Assistance to Motorists, or BOLO/APB/Warrants

In [268]:
# if outcome is in list, change flag from 0 to 1
reason_map = {
    'Moving Violation': 'moving',
    'Mechanical or Non-Moving Violation (V.C.)': 'mech_nonmoving',
    'DUI Check': 'dui',
    'Traffic Collisions': 'collision',
    'Assistance to Motorist': 'motor_assist',
    'MPC Violation': 'mpc',
    'BOLO/APB/Warrant': 'bolo'
}

for col in reason_map.values():
    stops_df[col] = 0

for reason, col in reason_map.items():
    stops_df.loc[stops_df['reason_for_stop'] == reason, col] = 1

stops_df = stops_df.drop(columns='reason_for_stop')

In [269]:
# remove nan values 
stops_df = stops_df.dropna()

# Imputing police districts 
- It may also be helpful to know the district a stop occurred in.
- To determine which district an observation belongs to, we first load in data from [San Francisco's Open Data Portal on police districts](https://data.sfgov.org/-/Map-of-Current-Police-Districts/wkhw-cjsf).

In [270]:
district_path = r'../data/police_districts.geojson'

districts = load_data(district_path)
districts

# add features of interest to stop observations
district_features = ['district', 'geometry']
districts = districts[district_features]

# check that stop data is in geopandas format
stops_df = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df.lng, stops_df.lat),
    crs="EPSG:4326"
)

# add community of concern data to observations
stops_df = gpd.sjoin(stops_df, districts, how="inner", predicate="within")
# drop unnecessary columns
unnecessary_cols = ['geometry', 'index_right']
stops_df = stops_df.drop(columns = unnecessary_cols)

# Adding Communities of Concern Flag
- Now we want to add a feature to flag whether a stop occurs in a Community of concern.
- We load in data from the [Metropolitan Transportation Commission](https://opendata-mtc.opendata.arcgis.com/datasets/MTC::equity-priority-communities-plan-bay-area-2040/about) that contains boundaries for communities of concern across the Bay Area.
- Here we alter the Equal Priority Community (epc, synonymous with Community of Concern) Class Flag to Change NA values to 'None' if an area is not an epc
- We also add data on the demographics of the community including the proportion of the population who are people of color, disabled, or 75+.

In [271]:
# Load data for communities of concern
concern_path = r"../data/communities_of_concern.geojson"
concern_df = load_data(concern_path)

# change epc_class to make NA = None
concern_df['epc_class'] = concern_df['epc_class'].replace({'NA':'Non-EPC'})

# add features of interest to collision events
concern_features = ['pct_over75', 'pct_poc', 'pct_disab', 'epc_class', 'geometry']
concern_df = concern_df[concern_features]

# check that stop data is in geopandas format
stops_df = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df.lng, stops_df.lat),
    crs="EPSG:4326"
)

# add community of concern data to observations
stops_df = gpd.sjoin(stops_df, concern_df, how="inner", predicate="within")

# drop unnecessary columns
unnecessary_cols = ['geometry', 'index_right']
stops_df = stops_df.drop(columns = unnecessary_cols)


# Create Day/Night Indicator
- To simplify times from 24 hour timestamps to visibility levels

### Function to get the sunrise, sunset, dawn, and dusk periods for a city

In [ ]:
from astral import LocationInfo
from astral.sun import sun
import pandas as pd
from datetime import date, timedelta

# add timezones to datetime for each observation
stops_df['datetime'] = pd.to_datetime(stops_df['date'].astype(str) + " " + stops_df['time'].astype(str))
stops_df['datetime'] = stops_df['datetime'].dt.tz_localize("America/Los_Angeles", ambiguous="NaT", nonexistent="shift_forward")
stops_df = stops_df.dropna(subset=['datetime'])

# function to get conditions for each day
def get_sun_df(start_date, end_date):
    city = LocationInfo("San Francisco", "USA", "America/Los_Angeles", 37.7749, -122.4194)
    rows = []
    current = start_date
    while current <= end_date:
        s = sun(city.observer, date=current)
        rows.append({
            "date": pd.Timestamp(current),
            "dawn": s["dawn"],
            "sunrise": s["sunrise"],
            "sunset": s["sunset"],
            "dusk": s["dusk"]
        })
        current += pd.Timedelta(days=1)
    return pd.DataFrame(rows)

sun_df = get_sun_df(stops_df['date'].min(), stops_df['date'].max())

# merge
stops_df = stops_df.merge(sun_df, on='date', how='left')
for col in ['dawn', 'sunrise', 'sunset', 'dusk']:
    stops_df[col] = stops_df[col].dt.tz_convert("America/Los_Angeles")
    # If sun time ends up before midnight of the date, shift to correct day
    stops_df[col] = stops_df.apply(
        lambda row: row[col] + pd.Timedelta(days=1) if row[col].date() < row['date'].date() else row[col],
        axis=1
    )

# assign condition based on datetime
conditions = [
    (stops_df['datetime'] >= stops_df['dawn']) & (stops_df['datetime'] < stops_df['sunrise']),  # dawn
    (stops_df['datetime'] >= stops_df['sunrise']) & (stops_df['datetime'] <= stops_df['sunset']), # day
    (stops_df['datetime'] > stops_df['sunset']) & (stops_df['datetime'] <= stops_df['dusk']),    # dusk
    (stops_df['datetime'] < stops_df['dawn']) | (stops_df['datetime'] > stops_df['dusk'])        # night
]
choices = ["dawn", "day", "dusk", "night"]
stops_df['light_condition'] = pd.Categorical(
    np.select(conditions, choices, default="night"),
    categories=["night", "dawn", "day", "dusk"],
    ordered=True
)

,date,time,lat,lng,subject_age,subject_race,subject_sex,arrest_made,citation_issued,warning_issued,...,pct_over75,pct_poc,pct_disab,epc_class,datetime,dawn,sunrise,sunset,dusk,light_condition
0,2010-01-01,00:10:00,37.744642,-122.424567,34.0,other,female,False,True,False,...,0.039024,0.250610,0.056098,Non-EPC,2010-01-01 00:10:00-08:00,2010-01-01 06:55:36.504172-08:00,2010-01-01 07:25:26.379594-08:00,2010-01-01 17:00:40.522244-08:00,2010-01-01 17:30:32.135176-08:00,night
1,2010-01-01,00:10:00,37.782744,-122.406499,22.0,other,male,False,True,False,...,0.032678,0.726959,0.228348,High,2010-01-01 00:10:00-08:00,2010-01-01 06:55:36.504172-08:00,2010-01-01 07:25:26.379594-08:00,2010-01-01 17:00:40.522244-08:00,2010-01-01 17:30:32.135176-08:00,night
2,2010-01-01,01:00:00,37.763301,-122.480603,27.0,asian/pacific islander,female,False,True,False,...,0.053179,0.651997,0.095762,Non-EPC,2010-01-01 01:00:00-08:00,2010-01-01 06:55:36.504172-08:00,2010-01-01 07:25:26.379594-08:00,2010-01-01 17:00:40.522244-08:00,2010-01-01 17:30:32.135176-08:00,night
3,2010-01-01,01:00:00,37.770318,-122.452079,25.0,hispanic,female,False,False,True,...,0.017103,0.310707,0.075717,Non-EPC,2010-01-01 01:00:00-08:00,2010-01-01 06:55:36.504172-08:00,2010-01-01 07:25:26.379594-08:00,2010-01-01 17:00:40.522244-08:00,2010-01-01 17:30:32.135176-08:00,night
4,2010-01-01,10:00:00,37.765696,-122.470729,36.0,white,male,False,True,False,...,0.044363,0.542523,0.034408,Non-EPC,2010-01-01 10:00:00-08:00,2010-01-01 06:55:36.504172-08:00,2010-01-01 07:25:26.379594-08:00,2010-01-01 17:00:40.522244-08:00,2010-01-01 17:30:32.135176-08:00,day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514723,2016-06-01,09:00:00,37.752153,-122.481713,49.0,asian/pacific islander,female,False,True,False,...,0.072781,0.783482,0.076432,Non-EPC,2016-06-01 09:00:00-07:00,2016-06-01 05:18:10.157120-07:00,2016-06-01 05:49:28.698235-07:00,2016-06-01 20:25:27.451333-07:00,2016-06-01 20:56:46.031714-07:00,day
514724,2016-06-01,09:28:00,37.766596,-122.421080,31.0,white,male,False,False,True,...,0.054539,0.725457,0.241148,High,2016-06-01 09:28:00-07:00,2016-06-01 05:18:10.157120-07:00,2016-06-01 05:49:28.698235-07:00,2016-06-01 20:25:27.451333-07:00,2016-06-01 20:56:46.031714-07:00,day
514725,2016-06-01,09:29:00,37.731383,-122.441703,70.0,other,male,False,True,False,...,0.076398,0.563980,0.087291,Non-EPC,2016-06-01 09:29:00-07:00,2016-06-01 05:18:10.157120-07:00,2016-06-01 05:49:28.698235-07:00,2016-06-01 20:25:27.451333-07:00,2016-06-01 20:56:46.031714-07:00,day
514726,2016-06-01,09:30:00,37.786122,-122.412437,22.0,white,male,False,False,True,...,0.035983,0.581653,0.100830,Non-EPC,2016-06-01 09:30:00-07:00,2016-06-01 05:18:10.157120-07:00,2016-06-01 05:49:28.698235-07:00,2016-06-01 20:25:27.451333-07:00,2016-06-01 20:56:46.031714-07:00,day


In [273]:
# drop unnecessary columns
unnecessary_cols = ['sunrise', 'sunset', 'dawn', 'dusk', 'datetime']
stops_df = stops_df.drop(columns = unnecessary_cols)

In [274]:
from zipfile import ZipFile

## export new cleaned dataset
# Save CSV
csv_path = r"../data/stops_clean.csv"
stops_df.to_csv(csv_path, index=False)

# zip file 
zip_file_name = r"../data/stops_clean.csv.zip"

with ZipFile(zip_file_name, 'w') as zipf:
    zipf.write(csv_path, arcname="stops_clean.csv")